# End-to-End Computer Vision Project: Flower Recognition

**Task:** Image classification, recognise the type of a flower from a photo.

**Pipeline covered in this notebook**
1. Problem statement and goal
2. Data collection (download dataset)
3. Exploratory data analysis (EDA)
4. Data preparation (stratified train/val/test split, augmentation)
5. Baseline model: simple CNN from scratch
6. Improved model: transfer learning (MobileNetV2) + fine-tuning
7. Evaluation: accuracy, classification report, confusion matrix, error analysis
8. Save the model and run inference on your own image
9. Deployment: Streamlit web app with a public link

> Use a GPU: *Runtime > Change runtime type > T4 GPU*, then run the cells from top to bottom.

## 1. Problem statement
Given a colour photo of a flower, predict which of 5 classes it belongs to
(daisy, dandelion, roses, sunflowers, tulips).

**Success metric:** accuracy on a held-out test set (images the model never saw during training),
plus per-class precision / recall to see which flowers are confused.

**Dataset:** the public *TensorFlow Flowers* dataset, about 3,670 photos in 5 folders (one folder per class).

## 2. Setup and data collection

In [ ]:
%pip install -q streamlit

import os, json, random, tarfile, urllib.request
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from PIL import Image

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
URL = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
DATA_ROOT = "/content/data"
DATA_DIR = os.path.join(DATA_ROOT, "flower_photos")

if not os.path.isdir(DATA_DIR):
    os.makedirs(DATA_ROOT, exist_ok=True)
    archive, _ = urllib.request.urlretrieve(URL, "flower_photos.tgz")
    with tarfile.open(archive) as tar:
        try:
            tar.extractall(DATA_ROOT, filter="data")
        except TypeError:          # older Python without the filter argument
            tar.extractall(DATA_ROOT)

class_names = sorted(d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d)))
print("Classes:", class_names)

## 3. Exploratory data analysis (EDA)

In [ ]:
# Collect every image path with its label
paths, labels = [], []
for idx, cname in enumerate(class_names):
    folder = os.path.join(DATA_DIR, cname)
    for f in os.listdir(folder):
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            paths.append(os.path.join(folder, f))
            labels.append(idx)
paths = np.array(paths); labels = np.array(labels)
print("Total images:", len(paths))

# 3.1 Class balance
counts = Counter(labels)
plt.figure(figsize=(6, 3))
plt.bar(class_names, [counts[i] for i in range(len(class_names))], color="#4c78a8")
plt.title("Images per class"); plt.ylabel("count"); plt.show()

# 3.2 Sample images
plt.figure(figsize=(12, 5))
for i, cname in enumerate(class_names):
    cls_paths = paths[labels == i]
    for j in range(3):
        plt.subplot(3, len(class_names), j * len(class_names) + i + 1)
        plt.imshow(Image.open(random.choice(cls_paths)))
        if j == 0: plt.title(cname)
        plt.axis("off")
plt.show()

# 3.3 Image sizes (sample of 300 images)
sizes = [Image.open(p).size for p in np.random.choice(paths, 300, replace=False)]
w, h = zip(*sizes)
print(f"Width : min {min(w)}, max {max(w)}, mean {np.mean(w):.0f}")
print(f"Height: min {min(h)}, max {max(h)}, mean {np.mean(h):.0f}")
print("Images have different sizes, so we resize everything to a fixed size.")

## 4. Data preparation
- Stratified split: **70% train / 15% validation / 15% test** (each class keeps the same proportion in every split).
- Resize to 224x224.
- Data augmentation (flip, rotation, zoom) is applied inside the model, only during training.

In [ ]:
from sklearn.model_selection import train_test_split

train_p, temp_p, train_y, temp_y = train_test_split(
    paths, labels, test_size=0.30, stratify=labels, random_state=SEED)
val_p, test_p, val_y, test_y = train_test_split(
    temp_p, temp_y, test_size=0.50, stratify=temp_y, random_state=SEED)
print(f"train {len(train_p)} | val {len(val_p)} | test {len(test_p)}")

IMG_SIZE = (224, 224)
BATCH = 32
AUTOTUNE = tf.data.AUTOTUNE

def decode(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)          # float32, values 0-255
    img.set_shape(IMG_SIZE + (3,))
    return img, label

def make_ds(p, y, training=False):
    ds = tf.data.Dataset.from_tensor_slices((p, y))
    if training:
        ds = ds.shuffle(len(p), seed=SEED)
    return ds.map(decode, AUTOTUNE).batch(BATCH).prefetch(AUTOTUNE)

train_ds = make_ds(train_p, train_y, training=True)
val_ds = make_ds(val_p, val_y)
test_ds = make_ds(test_p, test_y)
num_classes = len(class_names)

## 5. Baseline model: CNN from scratch
A small CNN trained from scratch gives us a reference point to compare against.

In [ ]:
baseline = models.Sequential([
    layers.Input(shape=IMG_SIZE + (3,)),
    layers.Rescaling(1.0 / 255),
    layers.RandomFlip("horizontal"),
    layers.Conv2D(32, 3, activation="relu"), layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu"), layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation="relu"), layers.MaxPooling2D(),
    layers.Dropout(0.3),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(num_classes),          # logits
], name="baseline_cnn")

baseline.compile(optimizer="adam",
                 loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                 metrics=["accuracy"])
hist_base = baseline.fit(train_ds, validation_data=val_ds, epochs=8)
base_loss, base_acc = baseline.evaluate(test_ds, verbose=0)
print(f"\nBaseline test accuracy: {base_acc:.4f}")

## 6. Improved model: transfer learning with MobileNetV2
MobileNetV2 is pretrained on ImageNet (1.4M images). We keep its learned visual features, add our own
classifier head, train the head first, then **fine-tune** the top layers with a very small learning rate.

In [ ]:
augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.15),
], name="augmentation")

base = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet")
base.trainable = False

inputs = layers.Input(shape=IMG_SIZE + (3,))
x = augment(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)   # scales to [-1, 1]
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes)(x)                        # logits
model = models.Model(inputs, outputs, name="flower_mobilenetv2")

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=loss_fn, metrics=["accuracy"])

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2),
]

# Stage 1: train only the new head
hist_head = model.fit(train_ds, validation_data=val_ds, epochs=8, callbacks=callbacks)

# Stage 2: fine-tune the top 30 layers of MobileNetV2
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss=loss_fn, metrics=["accuracy"])
hist_ft = model.fit(train_ds, validation_data=val_ds, epochs=6, callbacks=callbacks)

### Training curves

In [ ]:
acc = hist_head.history["accuracy"] + hist_ft.history["accuracy"]
val_acc = hist_head.history["val_accuracy"] + hist_ft.history["val_accuracy"]
loss = hist_head.history["loss"] + hist_ft.history["loss"]
val_loss = hist_head.history["val_loss"] + hist_ft.history["val_loss"]
split = len(hist_head.history["accuracy"])

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for a, tr, va, name in [(ax[0], acc, val_acc, "Accuracy"), (ax[1], loss, val_loss, "Loss")]:
    a.plot(tr, label="train"); a.plot(va, label="val")
    a.axvline(split - 1, color="gray", ls="--", label="fine-tuning starts")
    a.set_title(name); a.set_xlabel("epoch"); a.legend()
plt.show()

## 7. Evaluation on the unseen test set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

test_loss, test_acc = model.evaluate(test_ds, verbose=0)

print(pd.DataFrame({
    "Model": ["Baseline CNN (from scratch)", "MobileNetV2 (transfer learning)"],
    "Test accuracy": [f"{base_acc:.4f}", f"{test_acc:.4f}"],
}).to_string(index=False))

# predictions on the whole test set (test_ds is not shuffled, so order matches test_y)
probs = tf.nn.softmax(model.predict(test_ds, verbose=0), axis=1).numpy()
y_pred = probs.argmax(axis=1)

print("\n", classification_report(test_y, y_pred, target_names=class_names, digits=3))

fig, ax = plt.subplots(figsize=(5.5, 5))
ConfusionMatrixDisplay(confusion_matrix(test_y, y_pred), display_labels=class_names).plot(
    ax=ax, xticks_rotation=45, colorbar=False, cmap="Blues")
plt.title("Confusion matrix (test set)"); plt.show()

### Error analysis: what does the model get wrong?

In [ ]:
wrong = np.where(y_pred != test_y)[0]
print(f"{len(wrong)} wrong predictions out of {len(test_y)}")

plt.figure(figsize=(13, 6))
for k, i in enumerate(wrong[:10]):
    plt.subplot(2, 5, k + 1)
    plt.imshow(Image.open(test_p[i]))
    plt.title(f"pred: {class_names[y_pred[i]]}\ntrue: {class_names[test_y[i]]}\n({probs[i].max()*100:.0f}%)",
              fontsize=9, color="red")
    plt.axis("off")
plt.tight_layout(); plt.show()

## 8. Save the model and predict on your own image

In [ ]:
model.save("flower_model.keras")
with open("class_names.json", "w") as f:
    json.dump(class_names, f)
print("Saved flower_model.keras and class_names.json")

def predict_image(path, top=3):
    img = tf.keras.utils.load_img(path, target_size=IMG_SIZE)
    arr = tf.expand_dims(tf.keras.utils.img_to_array(img), 0)
    p = tf.nn.softmax(model.predict(arr, verbose=0)[0]).numpy()
    plt.imshow(img); plt.axis("off")
    plt.title(f"{class_names[p.argmax()]} ({p.max()*100:.1f}%)"); plt.show()
    for i in np.argsort(p)[::-1][:top]:
        print(f"{class_names[i]:<12} {p[i]*100:6.2f}%")

# Upload any flower photo from your computer
from google.colab import files
up = files.upload()
for name in up:
    predict_image(name)

## 9. Deployment: Streamlit web app

In [ ]:
%%writefile app.py
"""Streamlit app: Flower Recognition."""
import json

import numpy as np
import pandas as pd
import streamlit as st
import tensorflow as tf
from PIL import Image

MODEL_PATH = "flower_model.keras"
IMG_SIZE = (224, 224)

with open("class_names.json") as f:
    CLASS_NAMES = json.load(f)

st.set_page_config(page_title="Flower Recognition", page_icon="🌸")


@st.cache_resource(show_spinner="Loading model...")
def load_model():
    return tf.keras.models.load_model(MODEL_PATH)


def predict(model, image):
    # The model contains its own preprocessing, so we pass raw 0-255 pixels
    img = image.convert("RGB").resize(IMG_SIZE, Image.BILINEAR)
    arr = np.expand_dims(np.array(img, dtype="float32"), axis=0)
    return tf.nn.softmax(model.predict(arr, verbose=0)[0]).numpy()


st.title("🌸 Flower Recognition")
st.write("Upload a flower photo (or take one) and the deep learning model will tell you "
         f"which of the {len(CLASS_NAMES)} flower types it is: " + ", ".join(CLASS_NAMES) + ".")

model = load_model()

tab_up, tab_cam = st.tabs(["Upload image", "Use camera"])
with tab_up:
    file = st.file_uploader("Choose an image", type=["jpg", "jpeg", "png", "webp"])
with tab_cam:
    cam = st.camera_input("Take a photo")

source = file or cam
if source is not None:
    image = Image.open(source)
    score = predict(model, image)
    order = np.argsort(score)[::-1]

    c1, c2 = st.columns(2)
    c1.image(image, caption="Your image", use_container_width=True)
    c2.metric("Prediction", CLASS_NAMES[order[0]].title())
    c2.write(f"Confidence: **{score[order[0]] * 100:.2f}%**")
    if score[order[0]] < 0.6:
        c2.info("Low confidence: try a clearer photo of a single flower.")

    st.subheader("All class probabilities")
    st.bar_chart(pd.DataFrame({"Confidence (%)": score * 100}, index=CLASS_NAMES))

Colab runs on a remote machine, so `localhost` can't be opened directly. The next cell starts Streamlit
and a free Cloudflare tunnel. **Click the printed `trycloudflare.com` link** to open the app.

In [ ]:
import re, subprocess, sys, time

if not os.path.exists("cloudflared"):
    subprocess.run(["wget", "-q",
                    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
                    "-O", "cloudflared"], check=True)
    os.chmod("cloudflared", 0o755)

st_proc = subprocess.Popen([sys.executable, "-m", "streamlit", "run", "app.py",
                            "--server.port", "8501", "--server.headless", "true"])
time.sleep(8)

cf_log = open("cf.log", "w")
cf_proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
                           stdout=cf_log, stderr=subprocess.STDOUT)

public_url = None
for _ in range(30):
    time.sleep(2)
    m_ = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("cf.log").read())
    if m_:
        public_url = m_.group(0)
        break
print("Open your app here:", public_url if public_url else "not ready yet, run: print(open('cf.log').read())")

### Stop the app when finished

In [ ]:
cf_proc.terminate()
st_proc.terminate()
print("Stopped.")

## 10. Conclusion
- A baseline CNN trained from scratch gives a reasonable starting point, but **transfer learning with MobileNetV2** is clearly better
  because the pretrained features already understand edges, textures and shapes.
- Fine-tuning the top layers with a tiny learning rate gives a further small improvement.
- Error analysis shows which flower types get confused (see the confusion matrix and the misclassified images).
- The model is deployed as a **Streamlit app** that accepts uploads or camera photos.

**Possible improvements:** more data, stronger augmentation, a bigger backbone (EfficientNet), Grad-CAM to visualise what the model looks at,
and deploying permanently on Streamlit Community Cloud or Hugging Face Spaces.